# 04 · The study assistant — a LangGraph workflow

**Step 4 of the Study Assistant project.** This puts Steps 1–3 together into the assistant from the proposal. It answers from the course materials with citations, says *"This isn't covered in the course materials."* instead of guessing, explains topics simply, and makes practice quizzes.

It is a LangGraph `StateGraph`, the same pattern as `campus_help_desk_rag.ipynb` from class (router → nodes → conditional edges). Code: `src/study_assistant/assistant.py`, prompts in `prompts.py`.

```
question → route ─► chitchat (canned reply)
             │ answer / simplify / quiz
          retrieve   top 8 chunks, grouped into ≤5 passages (one per document)
             │
           check     off-topic floor (similarity < 0.25) + LLM relevance check
             ├── nothing relevant ─► abstain
          generate   answer / simple explanation / quiz, citing passages as [n]
             │
            cite     drop citations to passages that weren't shown, renumber,
                     map [n] → "Lecture 9 · Module 9 …, slide 6";
                     "not covered" reply → abstain; no citations → warning
```

**Why two safety layers?** Step 2 showed similarity scores can't separate covered from not-covered questions. So an LLM checks relevance, and the answering prompt *also* tells the model to reply "not covered" when the passages aren't enough.

From the terminal: `uv run python -m study_assistant.chat -v`

In [1]:
import warnings; warnings.filterwarnings("ignore")
from study_assistant.assistant import StudyAssistant
from study_assistant.llm import get_llm

bot = StudyAssistant(get_llm("ollama"))   # Llama 3.1 8B, local; dense retriever
print(bot.mermaid())

def show(question):
    r = bot.ask(question)
    print("\n".join(f"  · {line}" for line in r.trace))
    print("\n" + r.format())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	route(route)
	chitchat(chitchat)
	retrieve(retrieve)
	check(check)
	abstain(abstain)
	generate(generate)
	cite(cite)
	__end__([<p>__end__</p>]):::last
	__start__ --> route;
	check -.-> abstain;
	check -.-> generate;
	generate --> cite;
	retrieve --> check;
	route -.-> chitchat;
	route -.-> retrieve;
	abstain --> __end__;
	chitchat --> __end__;
	cite --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



## 1. An in-scope question
The relevance check keeps the three Lecture 9 slides and drops the unrelated VAE notebook; the answer cites the slide it used.

In [2]:
show("Why do transformers need positional encodings?")

W1124 16:10:17.004000 73835 /Users/apple/Desktop/ /Columbia/Sem 3/Assignments/Applied Gen AI/Final Project/.venv.nosync/lib/python3.12/site-packages/torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9120.75it/s]

  · route: mode=answer, search='Why do transformers need positional encodings?'
  · retrieve (dense): 8 chunks -> 5 passages
  ·   [1] 0.63 Lecture 9 · Module 9 – Transformer Architecture and Advanced GAN, slide 7
  ·   [2] 0.59 Lecture 9 · Module 9 – Transformer Architecture and Advanced GAN, slide 6
  ·   [3] 0.54 Lecture 9 · Module 9 – Transformer Architecture and Advanced GAN, slide 8
  ·   [4] 0.39 Lecture 5 · Module 4: Practical - Variational Autoencoders, cells 33-35 (Variational Autoencoder (VAE) Architecture)
  ·   [5] 0.39 Lecture 9 · Module 9 – Transformer Architecture and Advanced GAN, slide 2
  · check: grader says relevant=[1, 2, 3]
  · generate (ollama/llama3.1:8b, 5.2s)
  · cite: 1 source(s) cited

Transformers need positional encodings because the self-attention mechanism in transformers is invariant to the order of the input tokens. This means that the model cannot distinguish between tokens based on their position in the sequence. Positional encodings address this is

## 2. Out of scope: a near-miss and an obvious one
*Mode collapse* sounds like GAN material, but the course never covers it. *Capital of France* is plainly off-topic. Watch the `check` line: Llama 3.1 8B, acting as the grader, marks an irrelevant passage as relevant in both cases. The **second layer** (the answering prompt) catches it and the assistant abstains.

In [3]:
show("What is mode collapse in GANs and how do I fix it?")

  · route: mode=answer, search='What is mode collapse in GANs and how do I fix it?'
  · retrieve (dense): 8 chunks -> 5 passages
  ·   [1] 0.48 Lecture 9 · Module 9 – Transformer Architecture and Advanced GAN, slide 14
  ·   [2] 0.46 Lecture 9 · Module 9 – Transformer Architecture and Advanced GAN, slide 15
  ·   [3] 0.41 Lecture 6 · Module 6 Activity – Adding GAN to Helper Library, page 2
  ·   [4] 0.41 Lecture 6 · Module 6 – Modern Deep Learning Architectures Part 3, slide 13
  ·   [5] 0.39 Lecture 6 · Module 6: Practical - GAN, cells 7-9 (We perform the usual image transformations, but this time w…)
  · check: grader says relevant=[3]
  · generate (ollama/llama3.1:8b, 2.1s)
  · cite: model answered 'not covered' -> abstain

This isn't covered in the course materials.


In [4]:
show("What is the capital of France?")

  · route: mode=answer, search='What is the capital of France?'
  · retrieve (dense): 8 chunks -> 5 passages
  ·   [1] 0.36 Add-on lab · LLM / Agent Evaluations, cells 9-10 (Tier 3 — Semantic similarity (sentence embeddings))
  ·   [2] 0.22 Lecture 2 · Module 2 – Basics of Generative AI, slide 18
  ·   [3] 0.21 Add-on lab · Fine-tuning T5 for Translation — and Measuring Whether It Helped, cells 6-7 (The preprocessing function needs to)
  ·   [4] 0.18 Lecture 1 · Module 1 – Introduction, slide 22
  ·   [5] 0.17 Add-on lab · RAG Primer (Fast) — Retrieval-Augmented Generation, cells 1-2 (Introduction)
  · check: grader says relevant=[5]
  · generate (ollama/llama3.1:8b, 1.7s)
  · cite: model answered 'not covered' -> abstain

This isn't covered in the course materials.


## 3. A failure: the right lecture is never retrieved
This is a paraphrase of RLHF (Lecture 11). In Step 2 dense retrieval ranked the right chunk 6th, so it is not among the 5 passages. The assistant *correctly* refuses to answer from the wrong material, but the result is a **false "not covered"**. This is a retrieval problem, to be measured in Step 5 and worked on in Step 6.

In [5]:
show("How do you teach a chatbot to prefer the answers people like?")

  · route: mode=answer, search='How do you teach a chatbot to prefer the answers people like?'
  · retrieve (dense): 8 chunks -> 5 passages
  ·   [1] 0.49 Lecture 3 · Module 3 – Applications of Generative AI, slide 19
  ·   [2] 0.48 Lecture 3 · Module 3 – Applications of Generative AI, slide 8
  ·   [3] 0.45 Add-on lab · LangGraph Agentic Workflow, cells 23-28 (▶️ How to run this notebook (Colab or local VS Code / Jupyter) › Let us try a bit more complex example with a smart orchestr…)
  ·   [4] 0.43 Lecture 3 · Prompting Strategies (Module 3 · Applications of GenAI), cells 1-9 (Introduction)
  ·   [5] 0.43 Lecture 1 · Module 1 – Introduction, slide 10
  · check: grader says relevant=[1, 2]
  · generate (ollama/llama3.1:8b, 2.5s)
  · cite: model answered 'not covered' -> abstain

This isn't covered in the course materials.


## 4. Explain simply
Mode detection is rule-based ("in simple terms", "ELI5", "simpler", …), like the class router: cheap and predictable. The topic words are stripped of the request phrasing before searching.

In [6]:
show("Can you explain the Bellman equation in simple terms?")

  · route: mode=simplify, search='Bellman equation'
  · retrieve (dense): 8 chunks -> 5 passages
  ·   [1] 0.68 Lecture 10 · Module 10 – Basics of Reinforcement Learning, slide 14
  ·   [2] 0.61 Lecture 10 · Module 9 — Practical: Basics of Reinforcement Learning, cells 13-14 (2. Planning when you do know the map › 2.2 The Bellman equation, seen on one cell)
  ·   [3] 0.45 Lecture 10 · Module 10 – Basics of Reinforcement Learning (extended deck), slide 25
  ·   [4] 0.40 Lecture 10 · Module 9 — Practical: Basics of Reinforcement Learning, cells 15-16 (2. Planning when you do know the map › 🔎 Check your understanding)
  ·   [5] 0.39 Lecture 10 · Module 10 – Basics of Reinforcement Learning (extended deck), slide 2
  · check: grader says relevant=[1, 2, 3, 4]
  · generate (ollama/llama3.1:8b, 15.5s)
  · cite: 3 source(s) cited

Let's break it down.

The Bellman equation is a way to figure out the best way to get from one state to another in a game or problem. It's like a map that shows you

## 5. Practice quiz

In [7]:
show("Quiz me on GANs")

  · route: mode=quiz, search='GANs'
  · retrieve (dense): 8 chunks -> 5 passages
  ·   [1] 0.59 Lecture 9 · Module 9 – Transformer Architecture and Advanced GAN, slide 15
  ·   [2] 0.58 Lecture 6 · Module 6 – Modern Deep Learning Architectures Part 3, slide 13
  ·   [3] 0.57 Lecture 6 · Module 6 – Modern Deep Learning Architectures Part 3, slide 4
  ·   [4] 0.54 Lecture 9 · Module 9 – Transformer Architecture and Advanced GAN, slide 14
  ·   [5] 0.53 Lecture 6 · Module 6 – Modern Deep Learning Architectures Part 3, slide 12
  · check: grader says relevant=[3, 4, 5]
  · generate (ollama/llama3.1:8b, 19.3s)
  · cite: 3 source(s) cited

Q1. What is the primary goal of the Generator (G) in a Generative Adversarial Network (GAN)?
A) To distinguish real data from fake data. B) To generate synthetic data from random noise. C) To increase the complexity of the network. D) To transform one-hot encoded labels to match the format of the input images.

Answer: B) To generate synthetic data from ra

## 6. Small talk (no retrieval, no LLM call)

In [8]:
show("hello")

  · route: mode=chitchat, search=''
  · chitchat: canned reply

Hi! I'm the study assistant for Applied Generative AI. Ask me about anything from the lectures, practicals or activities, ask me to explain a topic more simply, or ask for a practice quiz (e.g. "quiz me on GANs").


## What we learned (anecdotal, 7 examples; Step 5 measures it on 50 questions)

| Observation | Implication |
|---|---|
| Answers cite the right lecture/slide; citations are remapped to a clean `[1]..[n]` source list | Citation accuracy can be scored automatically in Step 5 |
| The Llama-8B relevance check sometimes marks irrelevant passages as relevant | Two layers needed; Step 5 measures how often each layer catches out-of-scope questions |
| A paraphrased in-scope question got a false "not covered" because retrieval missed the lecture | Retrieval recall is the bottleneck to watch; Step 6 candidates: more passages, query rewriting, section-level fusion |
| flan-t5 (tried from the CLI) returns fragments without citations; the "uncited" warning flags it | flan uses a similarity threshold instead of the LLM check and stays as the weak baseline |